In [11]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from linearmodels.iv import IV2SLS
dataset = pd.read_csv(r'C:\Users\Spandan Dutta\econometrics_project\data\processed\master_panel.csv',parse_dates=['date'], index_col='date')

In [ ]:
#placebo test 1
control = ['d_us_10y', 'repo_rate', 'cpi']

# pre announcement 
pre = dataset[dataset.index < '2023-09-30'].copy()
pre_reg = pre[['d_yield_10y', 'fpi', 'jpmorgan_weight'] + control].dropna()
no_variation = pre_reg['jpmorgan_weight'].nunique() == 1
if no_variation:
    print('No variation in instrument')
else:
    X_pre = sm.add_constant(pre_reg[['jpmorgan_weight'] + control])
    first_stage_pre = sm.OLS(pre_reg['fpi'], X_pre).fit()
    print(first_stage_pre)
    if first_stage_pre.pvalues['jpmorgan_weight'] > 0.10:
        print('instrument not significant')
    else:
        print('anticipation effect')

No variation in instrument


In [32]:
#placebo test 2 : instrument is random
np.random.seed(42)
results = []
for i in range(50):
    df = dataset.copy()
    df['fake_weight'] = np.random.normal(0, 1,len(df))   #creating fake weight
    df_reg = df[['d_yield_10y', 'fpi', 'fake_weight'] + control].dropna()     
    X= sm.add_constant(df_reg[['fake_weight'] + control])
    first_stage = sm.OLS(df_reg['fpi'], X).fit() 
    f_test = first_stage.f_test("fake_weight = 0")
    f_stat = float(f_test.fvalue)
    p_val_instr = float(f_test.pvalue)
    #Second Stage
    X_ss =sm.add_constant(df_reg[control])
    model = IV2SLS(df_reg['d_yield_10y'],X_ss,df_reg[['fpi']],instruments=df_reg[['fake_weight']]).fit(cov_type='robust') 
    results.append({
        'Run': i + 1,
        'Coef': round(model.params['fpi'],4),
        'p-value': round(model.pvalues['fpi'],4),
        'F_stat (instrument)': round(f_stat,4),
        'p_value (instrument) ': round(p_val_instr,4)})
results_df = pd.DataFrame(results)
summary = results_df.agg({
    'Coef': ['mean', 'std'],
    'p-value': ['mean'],
    'F_stat (instrument)': ['mean'],
    'p_value (instrument) ': ['mean']})

print(summary)

          Coef   p-value  F_stat (instrument)  p_value (instrument) 
mean -0.430040  0.632902             1.269572               0.396894
std   2.571797       NaN                  NaN                    NaN
